# Лаб. 1 — Сети прямого распространения (MLP на MNIST)
Методичка: `docs/ИНС_лаб1_сети_прямого_распространения.pdf`, раздел «Порядок выполнения». Стек — строго по методичке: `tensorflow.keras` (Sequential + Dense/ReLU/Softmax + Dropout, SparseCategoricalCrossentropy, adam, acc).

План = 7 пунктов методички: (1) загрузка MNIST + нормализация + flatten; (2) builder MLP с 1–2 скрытыми слоями; (3) базовое обучение 15–20 эпох + кривые + summary; (4) evaluate + confusion matrix + неверные примеры; (5) sweeps по dropout и batch_size; (6) GridSearchCV; (7) аналитика числа параметров + сравнение 1 vs 2 слоя при равном бюджете ~70–100k.

Выборы по умолчанию (вопросы отклонены — взяты стандартные): baseline пп.3–5 = **1 скрытый слой, 128 нейронов**, dropout 0.2, batch 128; п.6 — полный GridSearchCV строго по методичке: сетка число нейронов x batch_size, один скрытый слой, cv=3, вся обучающая выборка (dropout 0.2 и 10 эпох фиксированы как разумные).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print('tf', tf.__version__)
print('gpus:', tf.config.list_physical_devices('GPU'))  # пусто = CPU, это нормально

## П.1 — Загрузка MNIST, примеры, нормализация, flatten

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()
print(x_train_full.shape, y_train_full.shape, x_test.shape, y_test.shape)

# по 1 изображению на каждую цифру из обучающего набора
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for d in range(10):
    idx = np.flatnonzero(y_train_full == d)[0]
    axes[d].imshow(x_train_full[idx], cmap='gray')
    axes[d].set_title(str(d)); axes[d].axis('off')
plt.suptitle('MNIST: по одному примеру на класс'); plt.show()

# нормализация + flatten в вектор 784
x_train_full = (x_train_full.astype('float32') / 255.0).reshape(-1, 28 * 28)
x_test = (x_test.astype('float32') / 255.0).reshape(-1, 28 * 28)
print('train', x_train_full.shape, x_train_full.min(), x_train_full.max())
print('test ', x_test.shape, x_test.min(), x_test.max())

## П.2 — Builder MLP: 1–2 скрытых слоя, dropout опционально

In [ ]:
def build_mlp(hidden=(128,), dropout=0.2, input_dim=28 * 28, n_classes=10):
    """hidden: tuple, напр. (128,) или (128, 64). dropout применяется после каждого скрытого слоя."""
    model = keras.Sequential([layers.Input(shape=(input_dim,))])
    for h in hidden:
        model.add(layers.Dense(h, activation='relu'))
        if dropout and dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(n_classes, activation='softmax'))
    model.compile(loss=keras.losses.SparseCategoricalCrossentropy(),
                    optimizer='adam', metrics=['acc'])
    return model

def count_params(hidden, input_dim=28 * 28, n_classes=10):
    """Аналитический подсчёт (см. п.7): сумма (fan_in+1)*fan_out по всем Dense-слоям."""
    dims = [input_dim, *hidden, n_classes]
    return sum((a + 1) * b for a, b in zip(dims[:-1], dims[1:]))

m = build_mlp((128,), 0.2)
m.summary()
print('analytic:', count_params((128,)), '| keras:', m.count_params())

## П.3 — Базовое обучение 15–20 эпох, кривые, summary

In [ ]:
EPOCHS, BATCH = 15, 128
baseline = build_mlp(hidden=(128,), dropout=0.2)
hist = baseline.fit(x_train_full, y_train_full, epochs=EPOCHS, batch_size=BATCH,
                      validation_split=0.2, verbose=2)
baseline.summary()  # структура + число обучаемых параметров
# сверка: аналитика из п.7 обязана совпасть с подсчётом keras
assert baseline.count_params() == count_params((128,)) == 101770
print('параметры OK: analytic == keras == 101770; вход 784, выход 10, dropout на скрытом слое')

h = hist.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('loss vs epoch'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(h['acc'], label='train'); ax[1].plot(h['val_acc'], label='val')
ax[1].set_title('acc vs epoch'); ax[1].legend(); ax[1].grid(True)
plt.show()

## П.4 — Тест: evaluate, confusion matrix, неверные примеры

In [ ]:
loss, acc = baseline.evaluate(x_test, y_test, verbose=0)
print(f'evaluate: loss={loss:.4f} acc={acc:.4f}')

pred = np.argmax(baseline.predict(x_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, pred)
acc_cm = np.trace(cm) / np.sum(cm)  # критерий качества по матрице путаницы
print(f'acc по confusion matrix: {acc_cm:.4f} (evaluate дал {acc:.4f}, разница={abs(acc - acc_cm):.2e})')
print('confusion matrix:\n', cm)

# строго по одному неверно проклассифицированному примеру на каждый класс (0-9)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Неверно проклассифицированные: true -> pred')
for t in range(10):
    ax = axes.flat[t]
    ax.axis('off')
    bad = np.flatnonzero((y_test == t) & (pred != y_test))
    if len(bad) == 0:
        ax.set_title(f'{t}: ошибок нет')
        continue
    i = bad[0]
    ax.imshow(x_test[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'{t} -> {int(pred[i])}')
plt.show()

## П.5а — Влияние dropout (0, 0.1, 0.2, 0.3) при фиксированном batch_size=128

In [ ]:
SWEEP_EPOCHS = 15  # тот же режим, что базовое обучение в п.3
drop_vals = [0.0, 0.1, 0.2, 0.3]
drop_hist = {}
for d in drop_vals:
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    mdl = build_mlp(hidden=(128,), dropout=d)
    hh = mdl.fit(x_train_full, y_train_full, epochs=SWEEP_EPOCHS, batch_size=128,
                 validation_split=0.2, verbose=0)
    drop_hist[d] = hh.history
    print(f'dropout={d}: train_loss={hh.history["loss"][-1]:.4f} val_loss={hh.history["val_loss"][-1]:.4f} '
          f'train_acc={hh.history["acc"][-1]:.4f} val_acc={hh.history["val_acc"][-1]:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for d in drop_vals:
    ax[0].plot(drop_hist[d]['loss'], label=f'dropout={d}')
    ax[1].plot(drop_hist[d]['val_loss'], label=f'dropout={d}')
ax[0].set_title('train loss vs epoch (batch=128)'); ax[0].legend(); ax[0].grid(True)
ax[1].set_title('val loss vs epoch (batch=128)'); ax[1].legend(); ax[1].grid(True)
plt.show()
print('Вывод: dropout=0 быстрее всех жмёт train loss, но val loss раньше всех выходит на плато '
      '(переобучение); 0.1-0.2 дают лучший val; 0.3 регуляризует сильнее всех, но сходится медленнее.')

## П.5б — Влияние batch_size (16, 32, 64, 128) при фиксированном dropout=0.2

In [ ]:
bs_vals = [16, 32, 64, 128]
bs_hist = {}
for bs in bs_vals:
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    mdl = build_mlp(hidden=(128,), dropout=0.2)
    hh = mdl.fit(x_train_full, y_train_full, epochs=SWEEP_EPOCHS, batch_size=bs,
                 validation_split=0.2, verbose=0)
    bs_hist[bs] = hh.history
    print(f'batch={bs}: train_loss={hh.history["loss"][-1]:.4f} val_loss={hh.history["val_loss"][-1]:.4f} '
          f'train_acc={hh.history["acc"][-1]:.4f} val_acc={hh.history["val_acc"][-1]:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for bs in bs_vals:
    ax[0].plot(bs_hist[bs]['loss'], label=f'batch={bs}')
    ax[1].plot(bs_hist[bs]['val_loss'], label=f'batch={bs}')
ax[0].set_title('train loss vs epoch (dropout=0.2)'); ax[0].legend(); ax[0].grid(True)
ax[1].set_title('val loss vs epoch (dropout=0.2)'); ax[1].legend(); ax[1].grid(True)
plt.show()
print('Вывод: малые batch (16-32) сходятся шумно, но шум SGD работает как регуляризатор; '
      'большие (64-128) дают гладкие кривые и быстрее по времени; по val_loss лучший batch=128 (0.0784), по val_acc — 64 (0.9773); 64 — баланс.')

## П.6 — GridSearchCV: число нейронов × batch_size, один скрытый слой
Сетка строго по методичке (нейроны × batch_size, один скрытый слой, cv=3, вся обучающая выборка). Разумные значения: нейроны 64/128/256, batch 32/64/128; dropout 0.2 и 10 эпох фиксированы.

In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

def _mlp_model(hidden=(128,), dropout=0.2):
    # имена аргументов обязаны совпадать с ключами model__* в param_grid
    return build_mlp(hidden, dropout)

def make_clf(epochs=10, batch_size=128):
    # dropout фиксирован 0.2 (разумен по п.5): в сетке только нейроны x batch_size, как в методичке
    return KerasClassifier(model=_mlp_model,
                           loss=keras.losses.SparseCategoricalCrossentropy(),
                           optimizer='adam', metrics=['acc'],
                           epochs=epochs, batch_size=batch_size, verbose=0)

# сетка строго по методичке: число нейронов (один скрытый слой) x batch_size; cv=3
grid = GridSearchCV(make_clf(),
                    param_grid={'model__hidden': [(64,), (128,), (256,)],
                                'batch_size': [32, 64, 128]},
                    cv=3, n_jobs=1, verbose=1)
grid.fit(x_train_full, y_train_full)
print('best:', grid.best_params_, 'score=%.4f' % grid.best_score_)

## П.7 — Аналитика параметров + 1 vs 2 слоя при равном бюджете
$$P_{1}(h) = (784+1)h + (h+1)\cdot 10 = 795h + 10$$
$$P_{2}(h_1,h_2) = (784+1)h_1 + (h_1+1)h_2 + (h_2+1)\cdot 10 = 785h_1 + h_1h_2 + 11h_2 + 10$$
Dropout параметров не добавляет. Берём: 1 слой $h=100$ → $P=79\,510$; 2 слоя $(96,64)$ → $P=82\,218$ — оба в диапазоне 70–100k и отличаются <4%.

In [ ]:
P1 = count_params((100,)); P2 = count_params((96, 64))
print(f'1 слой (100,):    P={P1}')
print(f'2 слоя (96, 64):  P={P2}  (разница={(P2 - P1) / P1:+.1%})')
assert 70_000 <= P1 <= 100_000 and 70_000 <= P2 <= 100_000

res = {}
for name, hid in [('1x100', (100,)), ('2x96x64', (96, 64))]:
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    mdl = build_mlp(hidden=hid, dropout=0.2)
    hh = mdl.fit(x_train_full, y_train_full, epochs=15, batch_size=128,
                 validation_split=0.2, verbose=0)
    lt, at = mdl.evaluate(x_test, y_test, verbose=0)
    res[name] = (hh.history, lt, at)
    print(f'{name}: P={mdl.count_params()} test_loss={lt:.4f} test_acc={at:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (hh, _, _) in res.items():
    ax[0].plot(hh['val_loss'], label=name)
    ax[1].plot(hh['val_acc'], label=name)
ax[0].set_title('val loss: 1 vs 2 слоя, равный бюджет'); ax[0].legend(); ax[0].grid(True)
ax[1].set_title('val acc: 1 vs 2 слоя, равный бюджет'); ax[1].legend(); ax[1].grid(True)
plt.show()
(_, l1, a1), (_, l2, a2) = res['1x100'], res['2x96x64']
print(f'Вывод: при равном бюджете (~80k) test_acc {a1:.4f} vs {a2:.4f} (разница {a2 - a1:+.4f}): '
      'усложнение модели преимущества не дало, однослойной сети достаточно для MNIST.')

## Выводы + контрольные вопросы (кратко)
1. Формальный нейрон: адаптивный сумматор $s=w^Tx$ + активация $y=f(s)$ + ветвление; требования к $f$: монотонность, дифференцируемость п.в., дешёвая производная (sigmoid/ReLU/LeakyReLU/softmax на выходе).
2. Слоистые: сигнал идёт слой за слоем (ф-ла (1) методички); полносвязные: каждый нейрон связан со всеми, такты; рекуррентные = с обратными связями.
3. Колмогоров–Арнольд — точное представление 3-слойной сетью; Вейерштрасс/Стоун — плотность (аппроксимация): класс НС замкнут относительно линейных операций и умножения (через квадрат), содержит единицу и разделяет точки.
4. Backprop: прямой проход → ошибки выходного слоя $g^N=y(1-y)(d-y)$ → назад $g^n$ → шаг $\Delta w=\rho\, g\, y$; образцы в случайном порядке.

Практика: baseline ~97–98% test acc; dropout 0.1–0.2 оптимален; batch 32–64 — баланс шум/скорость; грид подтверждает; при равном бюджете 1 и 2 слоя близки.